### 01: Склейка данных

Склейка `Cell_Avail`, `северо-запад_Rollout_RF-Site_vsf_20260224`, `NOC_TT_Report_23` (24 и 25 локально).  
Результат: `availability_with_alarms_merged` с колонками:  
RECDATE, Master Site, Region, Subregion,  
Cell Avail 2G (%), Cell Avail 3G (%), Cell Avail 4G (%), Cell Avail Base Tech (%),  
Номер сайта, Субрегион, Тип объекта, Адрес, Производитель БС,  
Широта WGS84, Долгота WGS84, Дата запуска, Высота подвеса, м,  
Alarm Count, Alarm Descriptions, Total Fault Time, Service

In [ ]:
df = df_cell_avail.copy()


In [ ]:
df_temp = df.copy()
df_temp['Master Site'] = df_temp['Master Site'].str.replace(r'([A-Za-z]+)00', r'\1', regex=True)

result = pd.merge(df_temp, df2, left_on='Master Site', right_on='Номер сайта', how='left')
result.head()

In [ ]:
df_avail = result

df_alarms['Alarm Up'] = pd.to_datetime(df_alarms['Alarm Up'], format='%m/%d/%y %H:%M', errors='coerce')
df_alarms['Closed Time'] = pd.to_datetime(df_alarms['Closed Time'], format='%m/%d/%y %H:%M', errors='coerce')
df_avail['RECDATE'] = pd.to_datetime(df_avail['RECDATE'])
df_avail['RECDATE_DATE'] = df_avail['RECDATE'].dt.date

nat_count = df_alarms['Alarm Up'].isna().sum()

df_alarms_clean = df_alarms.dropna(subset=['Alarm Up', 'Closed Time']).copy()

df_alarms_clean['Alarm description'] = df_alarms_clean['Alarm description'].fillna('Unknown').astype(str)
df_alarms_clean['Entry ID'] = df_alarms_clean['Entry ID'].fillna('Unknown').astype(str)
df_alarms_clean['Service'] = df_alarms_clean['Service'].fillna('Unknown').astype(str)
df_alarms_clean['Fault-time per TT'] = pd.to_numeric(df_alarms_clean['Fault-time per TT'], errors='coerce').fillna(0)

def get_affected_dates(alarm_up, closed_time):
    dates = []
    current_date = alarm_up.date()
    end_date = closed_time.date()

    while current_date <= end_date:
        dates.append(current_date)
        current_date += timedelta(days=1)

    return dates

expanded_alarms = []

for idx, row in df_alarms_clean.iterrows():
    affected_dates = get_affected_dates(row['Alarm Up'], row['Closed Time'])

    for date in affected_dates:
        alarm_copy = row.copy()
        alarm_copy['Affected Date'] = date
        expanded_alarms.append(alarm_copy)

df_expanded = pd.DataFrame(expanded_alarms)

alarm_summary = df_expanded.groupby(['Hostname', 'Affected Date']).agg({
    'Entry ID': 'count',
    'Alarm description': lambda x: ' | '.join(x.unique()),
    'Fault-time per TT': 'sum',
    'Service': 'first',
}).reset_index()

alarm_summary.columns = ['Master Site', 'RECDATE_DATE', 'Alarm Count', 'Alarm Descriptions', 'Total Fault Time', 'Service']


df_result = df_avail.merge(
    alarm_summary,
    left_on=['Master Site', 'RECDATE_DATE'],
    right_on=['Master Site', 'RECDATE_DATE'],
    how='left'
)

df_result['Alarm Count'] = df_result['Alarm Count'].fillna(0).astype(int)
df_result['Alarm Descriptions'] = df_result['Alarm Descriptions'].fillna('No alarms')
df_result['Total Fault Time'] = df_result['Total Fault Time'].fillna(0)
df_result['Service'] = df_result['Service'].fillna('Unknown')

df_final = df_result.drop(columns=['RECDATE_DATE'])
df_final.to_excel('availability_with_alarms_merged.xlsx', index=False)

In [ ]:
df_final.head()

,RECDATE,Master Site,Region,Subregion,Cell Avail 2G (%),Cell Avail 3G (%),Cell Avail 4G (%),Cell Avail Base Tech (%),Номер сайта,Субрегион,...,Адрес,Производитель БС,Широта WGS84,Долгота WGS84,Дата запуска,"Высота подвеса, м",Alarm Count,Alarm Descriptions,Total Fault Time,Service
0,2023-07-01,LE1715,SAINT PETERSBURG,СПб – Юг,NaN,100.00,100.00,NaN,LE1715,СПб – Юг,...,"Область Ленинградская, Район Тосненский, Посел...",Ericsson,59.730116,30.615187,2018-06-11,54.00,0,No alarms,0.00,Unknown
1,2023-07-01,SP0936,SAINT PETERSBURG,СПб – Юг,NaN,100.00,100.00,NaN,SP0936,СПб – Юг,...,"Город Санкт-Петербург, Проспект Славы, Дом 40,...",Ericsson,59.860988,30.397973,2019-07-23,27.00,0,No alarms,0.00,Unknown
2,2023-07-01,SP1781,SAINT PETERSBURG,СПб – Юг,NaN,100.00,100.00,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaT,NaN,0,No alarms,0.00,Unknown
3,2023-07-01,SP0342,SAINT PETERSBURG,СПб – Юг,84.65,84.68,83.33,83.99,SP0342,СПб – Юг,...,"Город Санкт-Петербург, Улица Курляндская, Дом ...",Ericsson,59.909617,30.280425,2019-05-14,14.00,1,FIRE,36.14,BTS
4,2023-07-01,LE1661,SAINT PETERSBURG,СПб – Юг,88.16,78.21,83.05,87.83,LE1661,СПб – Юг,...,"Область Ленинградская, Район Гатчинский, Масси...",Ericsson,59.527356,30.082492,2023-08-01,23.85,0,No alarms,0.00,Unknown
